# GraphMD — test split evaluation

Load the pretrained weights from training (`multiscale_mdg_nn_pretrained.pth` or a checkpoint), rebuild graphs from MISATO HDF5 using the **test** split, run inference, plot actual vs predicted ligand RMSD, and report **RMSE**.

**Match training:** use the same `N_FRAMES_PER_COMPLEX`, `POCKET_CUTOFF`, and `ATOM_FEATURE_DIM` as `graphmd-v2 (1).ipynb`.

In [ ]:
!pip install -q torch-geometric matplotlib

In [ ]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
import os, sys, subprocess

REPO_URL = "https://github.com/Vaheshan/GraphMD.git"
TARGET_DIR = "/kaggle/working/project"

if "KAGGLE_KERNEL_RUN_TYPE" in os.environ:
    if not os.path.exists(TARGET_DIR):
        print(f"Cloning repo from {REPO_URL} into {TARGET_DIR} ...")
        subprocess.run(["git", "clone", REPO_URL, TARGET_DIR], check=False)
    if TARGET_DIR not in sys.path:
        sys.path.append(TARGET_DIR)
    print("Repo path added to sys.path:", TARGET_DIR)
else:
    # Local: add this repo root if GraphMD lives alongside the notebook
    _here = os.path.abspath(os.getcwd())
    if _here not in sys.path:
        sys.path.insert(0, _here)
    print("Local mode: using sys.path[0] =", _here)

In [ ]:
import numpy as np
import h5py
import matplotlib.pyplot as plt
from typing import List, Dict, Any, Tuple

import torch
from torch import Tensor
from torch_geometric.data import Data

from graphs import (
    ProteinGraphBuilder,
    CorrelationEdgeBuilder,
    PocketGraphBuilder,
    ProteinGraphInputs,
    PocketGraphInputs,
)
from models import MultiscaleMDGNN
from training.batch_utils import collate_complexes

print("Imports done.")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# -------------------------------------------------------------------
# Paths — align with graphmd-v2 (1).ipynb
# -------------------------------------------------------------------
ROOT_DIR = '/kaggle/input/datasets/uom210636r/misato/'
RAW_DIR = ROOT_DIR

# Weights: final pretrained file from training cell (same Kaggle session / output dataset)
MODEL_PATH = '/kaggle/working/multiscale_mdg_nn_pretrained.pth'
# Optional: set to a checkpoint path to evaluate mid-training weights instead
CHECKPOINT_PATH = '/kaggle/working/multiscale_mdg_nn_checkpoint.pth'
LOAD_FROM_CHECKPOINT = False  # True -> load model_state_dict from checkpoint

# Must match training notebook
N_FRAMES_PER_COMPLEX = 5
POCKET_CUTOFF = 6.0
ATOM_FEATURE_DIM = 128
BATCH_SIZE = 2
MAX_COMPLEXES = None  # e.g. 50 for a quick test; None = all test IDs in each MD file

MD_HDF5_FILES = []
if os.path.exists(RAW_DIR):
    for fname in sorted(os.listdir(RAW_DIR)):
        if fname.lower().endswith('.hdf5') and fname.lower().startswith('md'):
            MD_HDF5_FILES.append(os.path.join(RAW_DIR, fname))

print('MD HDF5 files:')
for p in MD_HDF5_FILES:
    print(' ', p)

In [ ]:
# -------------------------------------------------------------------
# Utilities (same as training notebook)
# -------------------------------------------------------------------

def select_frame_indices(num_frames: int, n_select: int) -> List[int]:
    if n_select >= num_frames:
        return list(range(num_frames))
    idx = np.linspace(0, num_frames - 1, n_select, dtype=int)
    return sorted(set(idx.tolist()))


def load_split_ids(md_hdf5_path: str, split_type: str = "test") -> List[str]:
    base_name = os.path.splitext(os.path.basename(md_hdf5_path))[0]
    txt_file = os.path.join(
        RAW_DIR, "train_test_val", f"{base_name}_{split_type}.txt"
    )
    if not os.path.exists(txt_file):
        raise FileNotFoundError(f"No split file: {txt_file}")
    with open(txt_file, "r") as f:
        ids = [line.strip() for line in f if line.strip()]
    print(f"Loaded {len(ids)} PDB IDs ({split_type}) from {txt_file}")
    return ids


def build_residue_trajectories(
    misato_grp: h5py.Group,
    frame_indices: List[int],
) -> Tuple[Tensor, Tensor, np.ndarray]:
    coords_ds = misato_grp['trajectory_coordinates']
    T_full, num_atoms, _ = coords_ds.shape
    T_sel = len(frame_indices)

    ligand_begin = misato_grp['molecules_begin_atom_index'][:][-1]
    protein_len = int(ligand_begin)

    residue_idx = misato_grp['atoms_residue'][:protein_len]
    unique_res, inverse = np.unique(residue_idx, return_inverse=True)
    R = unique_res.shape[0]

    first_frame = frame_indices[0]
    coords0 = coords_ds[first_frame, :protein_len, :]

    sums0 = np.zeros((R, 3), dtype=np.float32)
    counts0 = np.zeros(R, dtype=np.int64)
    np.add.at(sums0, inverse, coords0)
    np.add.at(counts0, inverse, 1)
    centroids0 = sums0 / counts0[:, None]

    offset_N = np.array([0.5, 0.0, 0.0], dtype=np.float32)
    offset_C = np.array([0.0, 0.5, 0.0], dtype=np.float32)
    backbone_coords = np.stack(
        [centroids0 + offset_N, centroids0, centroids0 + offset_C],
        axis=1,
    )

    md_res = np.zeros((T_sel, R, 3), dtype=np.float32)
    for ti, frame_idx in enumerate(frame_indices):
        coords_t = coords_ds[frame_idx, :protein_len, :]
        sums_t = np.zeros((R, 3), dtype=np.float32)
        counts_t = np.zeros(R, dtype=np.int64)
        np.add.at(sums_t, inverse, coords_t)
        np.add.at(counts_t, inverse, 1)
        centroids_t = sums_t / counts_t[:, None]
        md_res[ti] = centroids_t

    atom_to_residue_full = -np.ones(num_atoms, dtype=np.int64)
    atom_to_residue_full[:protein_len] = inverse

    return (
        torch.from_numpy(backbone_coords).float(),
        torch.from_numpy(md_res).float(),
        atom_to_residue_full,
    )


def build_pocket_graphs_for_complex(
    misato_grp: h5py.Group,
    frame_indices: List[int],
    atom_to_residue_full: np.ndarray,
    pocket_builder: PocketGraphBuilder,
    pocket_cutoff: float,
    atom_feature_dim: int,
) -> Tuple[List[Data], Tensor]:
    traj = misato_grp['trajectory_coordinates'][frame_indices, :, :]
    T_sel, num_atoms, _ = traj.shape

    atom_numbers = misato_grp['atoms_number'][:]
    ligand_begin = misato_grp['molecules_begin_atom_index'][:][-1]

    ligand_mask_global = np.arange(num_atoms) >= ligand_begin
    heavy_atom_mask = atom_numbers != 1
    ligand_mask_global = ligand_mask_global & heavy_atom_mask

    rmsd_all = misato_grp['frames_rmsd_ligand'][:]
    y_stability = torch.tensor(rmsd_all[frame_indices], dtype=torch.float32)

    pocket_graphs: List[Data] = []
    for ti in range(T_sel):
        coords_t = traj[ti]
        ligand_indices = np.where(ligand_mask_global)[0]
        if ligand_indices.size == 0:
            pocket_indices = np.arange(num_atoms)
        else:
            ligand_coords = coords_t[ligand_indices]
            ligand_centroid = ligand_coords.mean(axis=0, keepdims=True)
            dists_to_ligand = np.linalg.norm(coords_t - ligand_centroid, axis=1)
            pocket_mask = dists_to_ligand <= pocket_cutoff
            pocket_indices = np.where(pocket_mask | ligand_mask_global)[0]

        coords_pocket = torch.from_numpy(coords_t[pocket_indices].astype(np.float32))
        atom_nums_pocket = atom_numbers[pocket_indices].astype(np.int64)

        atom_idx = atom_nums_pocket - 1
        atom_idx[atom_idx < 0] = 0
        atom_idx[atom_idx >= atom_feature_dim] = atom_feature_dim - 1
        atom_idx_t = torch.from_numpy(atom_idx)
        atom_features = torch.nn.functional.one_hot(
            atom_idx_t, num_classes=atom_feature_dim
        ).float()

        atom_is_ligand = torch.from_numpy(
            ligand_mask_global[pocket_indices].astype(np.bool_)
        )

        atom_to_residue = []
        for idx in pocket_indices:
            if ligand_mask_global[idx]:
                atom_to_residue.append(-1)
            else:
                atom_to_residue.append(int(atom_to_residue_full[idx]))
        atom_to_residue_t = torch.tensor(atom_to_residue, dtype=torch.long)

        pocket_inputs = PocketGraphInputs(
            atom_coords=coords_pocket,
            atom_features=atom_features,
            atom_is_ligand=atom_is_ligand,
            atom_to_residue=atom_to_residue_t,
        )
        pocket_graphs.append(pocket_builder(pocket_inputs))

    return pocket_graphs, y_stability


print("Utility functions defined.")

In [ ]:
# Frame selection (same rule as training: first file, first complex)
if not MD_HDF5_FILES:
    raise FileNotFoundError(f"No MD*.hdf5 under {RAW_DIR}")

md_example_path = MD_HDF5_FILES[0]
with h5py.File(md_example_path, "r") as f:
    pdb_ids = list(f.keys())
    first_id = pdb_ids[0]
    grp0 = f[first_id]
    num_frames = grp0["trajectory_coordinates"].shape[0]
    FRAME_INDICES = select_frame_indices(num_frames, N_FRAMES_PER_COMPLEX)
    print("Example complex:", first_id, "num_frames:", num_frames)
    print("FRAME_INDICES:", FRAME_INDICES)

In [ ]:
# Load model weights
model = MultiscaleMDGNN(atom_feature_dim=ATOM_FEATURE_DIM).to(device)

if LOAD_FROM_CHECKPOINT and os.path.exists(CHECKPOINT_PATH):
    ckpt = torch.load(CHECKPOINT_PATH, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
    print("Loaded model from checkpoint:", CHECKPOINT_PATH)
elif os.path.exists(MODEL_PATH):
    model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
    print("Loaded model from:", MODEL_PATH)
else:
    raise FileNotFoundError(
        f"No weights at {MODEL_PATH}. Add as Kaggle input or set LOAD_FROM_CHECKPOINT and checkpoint path."
    )

model.eval()

In [ ]:
# -------------------------------------------------------------------
# Evaluate on test split (streaming batches, same graph build as training)
# -------------------------------------------------------------------
from tqdm.auto import tqdm

y_true_all: List[float] = []
y_pred_all: List[float] = []

with torch.no_grad():
    for md_path in MD_HDF5_FILES:
        test_ids = load_split_ids(md_path, split_type="test")
        if MAX_COMPLEXES is not None:
            test_ids = test_ids[:MAX_COMPLEXES]

        protein_builder = ProteinGraphBuilder()
        corr_builder = CorrelationEdgeBuilder()
        pocket_builder = PocketGraphBuilder()

        with h5py.File(md_path, "r") as f:
            file_keys = set(f.keys())
            pending: List[Dict[str, Any]] = []

            def run_batch(items: List[Dict[str, Any]]) -> None:
                if not items:
                    return
                n_frames = len(items[0]["pocket_graphs"])
                for t_local in range(n_frames):
                    p_graphs = [it["protein_graph"] for it in items]
                    pk_graphs = [it["pocket_graphs"][t_local] for it in items]
                    y_t = torch.stack([it["y_stability"][t_local] for it in items]).float()
                    gbatch = collate_complexes(
                        p_graphs,
                        pk_graphs,
                        {"y_stability": y_t},
                    )
                    out = model(
                        {
                            "protein": gbatch.protein.to(device),
                            "pocket": gbatch.pocket.to(device),
                        }
                    )
                    pred = out["y_pred"].view(-1).detach().cpu().numpy()
                    true_ = y_t.numpy()
                    y_pred_all.extend(pred.tolist())
                    y_true_all.extend(true_.tolist())

            for pdb_id in tqdm(test_ids, desc=os.path.basename(md_path), leave=False):
                if pdb_id not in file_keys:
                    continue
                grp = f[pdb_id]
                bb, md_res, atom2res = build_residue_trajectories(grp, FRAME_INDICES)
                prot_in = ProteinGraphInputs(
                    backbone_coords=bb, md_residue_coords=md_res
                )
                pg = protein_builder(prot_in)
                pg = corr_builder.add_correlation_edges(pg, md_res)
                pocks, ystab = build_pocket_graphs_for_complex(
                    grp,
                    FRAME_INDICES,
                    atom2res,
                    pocket_builder,
                    POCKET_CUTOFF,
                    ATOM_FEATURE_DIM,
                )
                pending.append(
                    {"protein_graph": pg, "pocket_graphs": pocks, "y_stability": ystab}
                )
                if len(pending) >= BATCH_SIZE:
                    run_batch(pending)
                    pending = []
                    if torch.cuda.is_available():
                        torch.cuda.empty_cache()

            run_batch(pending)

y_true_arr = np.asarray(y_true_all, dtype=np.float64)
y_pred_arr = np.asarray(y_pred_all, dtype=np.float64)
print(f"Collected {len(y_true_arr)} frame-level predictions.")

rmse = float(np.sqrt(np.mean((y_true_arr - y_pred_arr) ** 2)))
print(f"RMSE (ligand RMSD): {rmse:.6f}")

In [ ]:
# Plot: actual vs predicted
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_true_arr, y_pred_arr, alpha=0.35, s=8, edgecolors="none")
lims = [
    min(y_true_arr.min(), y_pred_arr.min()),
    max(y_true_arr.max(), y_pred_arr.max()),
]
ax.plot(lims, lims, "r--", lw=1.5, label="y = x")
ax.set_xlabel("Actual frames_rmsd_ligand")
ax.set_ylabel("Predicted")
ax.set_title(f"Test set (N={len(y_true_arr)})  RMSE={rmse:.4f}")
ax.legend()
ax.set_aspect("equal", adjustable="box")
plt.tight_layout()
plt.show()

# Save figure (Kaggle working dir or current folder)
if "KAGGLE_KERNEL_RUN_TYPE" in os.environ:
    out_fig = "/kaggle/working/test_actual_vs_predicted.png"
else:
    out_fig = os.path.join(os.getcwd(), "test_actual_vs_predicted.png")
fig.savefig(out_fig, dpi=150)
print("Saved figure to", out_fig)